# Keep your code clean

Jupyter notebooks often start as exploration and experimentation by an indivudal data scientist. The code is not mean to be shared and certainly not to run beyond the early days of a project. However, snippets of code from Jupyter notebooks often make their way to production servers, where they run for months, possibly years.

Software engineers have been developing best practices around making code more readable. Although the basic principles are shared, languages also develop their own culsture and aesthetics. 

**NOTE**: The examples below are guidelines. There are often good reasons to deviate from best practices. Do not feel beholden to the rules below. As with any culture, some norms are barely enforced and some norm violations will cast you out.



### Python has its own naming conventions

For example, in Java variables are namde in camel case, such as `studentAge`, `dateOfBirth`, C# uses pascal case capitalizes the first letter, such as `StudentAge`, `DateOfBirth`, lisp uses kebab-case as `student-age`, `date-of-birth` and our favorite language, Python uses snake-case: `student_age`, `date_of_birth` for variables. 

Languages often use a different convention for class names. Python pascal case: `TextClassifier`, `ConsoleLogger`.

**Variables and functions should be snake_case**
```python

# Bad
ClientAge = 34 # Pascal case, not OK for variables, OK for classes
socialSecurityNumber = 1234567890 # Camel case, not used in Python
target-point = (34,54) # Not even synctatic Python

# Good
client_age = 34
social_security_number = 1234567890
target_point = (34,54)

NUM_OF_THREADS = 8 # For CONSTANTS, this "MACRO_CASE" or "CONSTANT_CASE" format is appropriate

def remove_punctuation(chat_sentence): pass # Notice that functions in Python are also snake case
```

**Class names should be PascalCase**
```python
class TextClasifier:
    pass

class ResNet(model):
    pass
```

### Names should be meaningful
**Variables**

```python

# Bad
x = np.mean(data)
df2 = process(df1)
lst = [1, 2, 3]

# Good
average_score = np.mean(student_scores)
processed_df = process(raw_dataframe)
prime_numbers = [1, 2, 3]
```

Notice that we can make this look even better (and easier to ready) by aligning the equals signs:
```python
average_score       = np.mean(student_scores)
processed_dataframe = process(raw_dataframe)
prime_numbers       = [1, 2, 3]
```


**Functions**

```python
# Bad
def proc_data():
    pass

# OK
def preprocess_text_data(raw_text):
    pass

# Good
def removes_punctuation(raw_text):
    pass
```

**Classes**

```python
# Bad
class Mdl:
    pass

# Good
class TextClassifierModel:
    pass
```

### Imports should be organized by source and functionality

```python
# Standard library
import os
from typing import Dict, List

# Third-party
import numpy as np
import pandas as pd
from sklearn.model_selection import train_test_split

# Local
from src.data.loading import load_dataset
from src.features.engineering import create_features
```

### Functions should be documented using docstrings

In [1]:
def calculate_feature_importance(
    model,
    feature_name
) :
    """Calculate feature importance scores from a trained random forest model.
    
    Args:
        model: Trained random forest classifier
        feature_names: List of feature names corresponding to model features
        
    Returns:
        Dictionary mapping feature names to their importance scores
        
    Raises:
        ValueError: If length of feature_names doesn't match model features
    """
    pass

Notice that this docstring now allows us to ask for help

In [2]:
help(calculate_feature_importance)

Help on function calculate_feature_importance in module __main__:

calculate_feature_importance(model, feature_name)
    Calculate feature importance scores from a trained random forest model.
    
    Args:
        model: Trained random forest classifier
        feature_names: List of feature names corresponding to model features
        
    Returns:
        Dictionary mapping feature names to their importance scores
        
    Raises:
        ValueError: If length of feature_names doesn't match model features



In [ ]:
calculate_feature_importance() #place cursor inside parenthesis and press SHIFT+TAB

### Functions should not be too large and do one thing

(docstrings elided from examples below to save space)

```python
# Bad
def analyze_market_volatility_regimes():

    # Load and validate tick data
    df = pd.read_csv('ticks.csv', parse_dates=['timestamp'])
    df = df.dropna()
    df = df[df['price'] > 0]
    if len(df) == 0:
        raise ValueError('No valid tick data after filtering')
    
    # Resample to daily OHLC
    daily = df.set_index('timestamp').groupby(
        pd.Grouper(freq='D')
    ).agg({
        'price': ['first', 'max', 'min', 'last'],
        'volume': 'sum'
    }).droplevel(0, axis=1)
    daily.columns = ['open', 'high', 'low', 'close', 'volume']
    
    # Compute returns and rolling volatility
    daily['returns'] = daily['close'].pct_change()
    daily['volatility'] = daily['returns'].rolling(20).std() * np.sqrt(252)
    daily['volatility_ma'] = daily['volatility'].rolling(5).mean()
    
    # Identify high-volatility regimes (above 90th percentile)
    vol_threshold = daily['volatility'].quantile(0.90)
    daily['high_vol_regime'] = daily['volatility'] > vol_threshold
    
    # Find contiguous high-volatility periods
    daily['regime_id'] = (daily['high_vol_regime'] != 
                          daily['high_vol_regime'].shift()).cumsum()
    regimes = daily[daily['high_vol_regime']].groupby('regime_id').agg({
        'volatility': ['min', 'max', 'mean'],
        'returns': ['min', 'max', 'std'],
        'close': 'count'  # duration in days
    })
    
    # Print summary statistics
    print(f"Number of high-volatility regimes: {len(regimes)}")
    print(f"Average regime duration: {regimes['close'].mean():.1f} days")
    print(f"Max volatility in any regime: {regimes['volatility']['max'].max():.2%}")
    
    return daily, regimes

# Good
def load_and_validate_ticks(filepath: str) -> pd.DataFrame:

    df = pd.read_csv(filepath, parse_dates=['timestamp'])
    df = df.dropna(subset=['price', 'volume'])
    df = df[df['price'] > 0]
    df = df.sort_values('timestamp')
    
    if len(df) == 0:
        raise ValueError('No valid tick data after filtering')
    
    return df


def compute_daily_ohlc(ticks: pd.DataFrame) -> pd.DataFrame:

    daily = ticks.set_index('timestamp').groupby(
        pd.Grouper(freq='D')
    ).agg({
        'price': ['first', 'max', 'min', 'last'],
        'volume': 'sum'
    }).droplevel(0, axis=1)
    daily.columns = ['open', 'high', 'low', 'close', 'volume']
    return daily


def compute_returns_and_volatility(daily: pd.DataFrame, 
                                     vol_window: int = 20) -> pd.DataFrame:

    daily = daily.copy()
    daily['returns'] = daily['close'].pct_change()
    daily['volatility'] = daily['returns'].rolling(vol_window).std() * np.sqrt(252)
    daily['volatility_ma'] = daily['volatility'].rolling(5).mean()
    return daily


def identify_volatility_regimes(daily: pd.DataFrame, 
                                 percentile: float = 0.90) -> pd.DataFrame:

    daily = daily.copy()
    vol_threshold = daily['volatility'].quantile(percentile)
    daily['high_vol_regime'] = daily['volatility'] > vol_threshold
    
    # Assign a unique ID to each contiguous regime
    daily['regime_id'] = (daily['high_vol_regime'] != 
                          daily['high_vol_regime'].shift()).cumsum()
    
    return daily


def summarize_regimes(daily: pd.DataFrame) -> pd.DataFrame:

    regimes = daily[daily['high_vol_regime']].groupby('regime_id').agg({
        'volatility': ['min', 'max', 'mean'],
        'returns': ['min', 'max', 'std'],
        'close': 'count'  # duration in days
    })
    return regimes


def print_volatility_report(daily: pd.DataFrame, regimes: pd.DataFrame) -> None:

    print(f"Number of high-volatility regimes: {len(regimes)}")
    print(f"Average regime duration: {regimes['close'].mean():.1f} days")
    print(f"Max volatility in any regime: {regimes['volatility']['max'].max():.2%}")
    print(f"\nDetailed regime summary:")
    print(regimes)


def analyze_market_volatility_regimes(filepath: str) -> pd.DataFrame:

    # Pipeline
    ticks = load_and_validate_ticks(filepath)
    daily = compute_daily_ohlc(ticks)
    daily = compute_returns_and_volatility(daily, vol_window=20)
    daily = identify_volatility_regimes(daily, percentile=0.90)
    regimes = summarize_regimes(daily)
    print_volatility_report(daily, regimes)
    
    return daily

```

### Be mindful of space

Give your code room to breathe!

Bad
```python
def analyze_market_volatility_regimes(filepath:str)->pd.DataFrame:
    # Pipeline
    ticks=load_and_validate_ticks(filepath)
    daily=compute_daily_ohlc(ticks)
    daily=compute_returns_and_volatility(daily,vol_window=20)
    daily=identify_volatility_regimes(daily,percentile=0.90)
    regimes = summarize_regimes(daily)
    print_volatility_report(daily, egimes)
    return daily
```

Good
```python
def analyze_market_volatility_regimes(filepath: str) -> pd.DataFrame:

    # Pipeline
    ticks = load_and_validate_ticks(filepath)
    daily = compute_daily_ohlc(ticks)
    daily = compute_returns_and_volatility(daily, vol_window=20)
    daily = identify_volatility_regimes(daily, percentile=0.90)
    regimes = summarize_regimes(daily)
    print_volatility_report(daily, regimes)
    
    return daily
```

### Explain the code appropriately

Bad: Explains what the code does

```python
# Loop through the dataframe
for idx, row in df.iterrows():
    processed.append(row)
```

Good: Explains why the code is needed

```python
# Handle missing values before modeling to prevent training errors
df.fillna(df.mean(), inplace=True)
```

**NOTE** We will learn more about these conventions, including additional ones, like adding type annotations, throughout this course.

### Automated tools
There are a two categories of tools relevant to the items discussed here: **linters** and **formatters**. 

Linters check the code for common formatting mistakes, but ones which don't elevant to the level of syntax errors. They pick up "dirt" the way a lint roller picks up lint from clothes.

Formatters, change your code so it matches a pre-defined format (for things like indentation, how code is spread across lines, etc.)

A common linter for python is `pylint` and a common formatter for python is a tool called `black`. Both are still widely used in the industry; however, a tool gaining popularity is `ruff`. This tool combines the features of linters and formatters.

In [ ]:
%%writefile messy.py
# messy.py
import pandas as pd,numpy as np
from typing import List,Dict,Any
import matplotlib.pyplot as plt

class dataProcessor:
    def __init__(self,input_file:str,    output_file:str='processed.csv'):
        self.input=input_file
        self.output_file=output_file 
        self.data=None
    
    def Load_data(self):
        """loads data from csv file"""
        self.data=pd.read_csv(self.input)
        return self.data
    
    def process(self,columns_to_process:List[str]=[],aggfunc:str='mean')->pd.DataFrame:
        if len(columns_to_process)==0: return self.data
        processed_data={}
        for col in columns_to_process:
         if col in self.data.columns:
          processed_data[col]=getattr(self.data[col],aggfunc)()
         else:
            print(f"Warning: Column {col} not found")
        return pd.DataFrame(processed_data,index=[0])

    def visualize_data(self,   column:str,   PlotType:str='bar'   )->None:
        if self.data is None:raise ValueError('No data loaded')
        plt.figure(figsize=(10,     5))
        if PlotType=='bar':
            self.data[column].value_counts().plot(kind='bar')
        elif     PlotType=='hist':
            self.data[column].hist()
        plt.title(f'Visualization of {column}')
        plt.show()

def main():
    processor=dataProcessor('data.csv')
    df = processor.Load_data()
    processed=processor.process(['age','salary'],aggfunc='mean')
    processor.visualize_data('age','hist')

if __name__=='__main__':
    main()

Turn on line numbers from View->Show Line Numbers to match output from the following programs to the program above

You may have to install these tools:

`pip install ruff pylint black`

In [ ]:
#!pip install ruff

In [ ]:
!ruff check messy.py

In [ ]:
!ruff format --diff messy.py 

NOTE: Exercise created with assistance from Claud AI